# Plotting Profiles for $\nu_{SC}$ and $\eta_c^r$

This notebook implements interactive visualizations for:

1. The semicircle distribution
$$
\nu_{\mathrm{SC}}(\mathrm dx)=\mathbf{1}_{|x|\le 2}\,\frac{\sqrt{4-x^2}}{2\pi}\,\mathrm dx.
$$

2. The family
$$
\mathrm d\eta_c^r = f_c^r\,\mathrm d\nu_{SC} + \mathrm d\eta_{c,\mathrm{sing}}^r,
$$
with
$$
f_c^r(x)=\mathbb E\!\left[\frac{1}{e^{-2c}R_r^2-xe^{-c}R_r+1}\right],\quad x\in(-2,2),
$$
and
$$
\eta_{c,\mathrm{sing}}^r(A)=\mathbb E\!\left[\left(1-e^{2c}R_r^{-2}\right)_+\mathbf{1}_A\!\left(e^{-c}R_r+e^cR_r^{-1}\right)\right].
$$

Here $R_r=\cos(T_r)$ with:
- $T_0=0$ (so $R_0=1$),
- $T_r\sim\mathcal N(0,2r)$ for $0<r<\infty$,
- $T_\infty\sim\mathrm{Unif}([0,2\pi])$.

In [1]:
import numpy as np
import plotly.graph_objects as go
from scipy.integrate import quad
from ipywidgets import FloatSlider, SelectionSlider, VBox, HBox, Output, Label
from IPython.display import display

In [2]:
# Numerical core: semicircle law, f_c^r, continuous density, singular approximation, TV distance

import warnings
from scipy.integrate import IntegrationWarning

def semicircle_density(x):
    out = np.zeros_like(x, dtype=float)
    mask = np.abs(x) <= 2.0
    out[mask] = np.sqrt(4.0 - x[mask] ** 2) / (2.0 * np.pi)
    return out


def _quad_safe(func, a, b, epsabs=1e-6, epsrel=1e-4, limit=80):
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', IntegrationWarning)
        val, _ = quad(func, a, b, epsabs=epsabs, epsrel=epsrel, limit=limit)
    return float(val)


def _f_single_x(x_val, c, r, epsabs=1e-6, epsrel=1e-4):
    if r == 0.0:
        den = np.exp(-2 * c) - x_val * np.exp(-c) + 1.0
        return 0.0 if abs(den) < 1e-12 else 1.0 / den

    if np.isinf(r):
        def integrand_inf(t):
            ct = np.cos(t)
            den = np.exp(-2 * c) * ct * ct - x_val * np.exp(-c) * ct + 1.0
            if abs(den) < 1e-12:
                return 0.0
            return 1.0 / (2.0 * np.pi * den)

        try:
            val = _quad_safe(integrand_inf, 0.0, 2.0 * np.pi, epsabs=epsabs, epsrel=epsrel, limit=120)
            return max(val, 0.0)
        except Exception:
            t = np.linspace(0.0, 2.0 * np.pi, 1200)
            ct = np.cos(t)
            den = np.exp(-2 * c) * ct * ct - x_val * np.exp(-c) * ct + 1.0
            vals = np.where(np.abs(den) < 1e-12, 0.0, 1.0 / den)
            return max(float(np.mean(vals)), 0.0)

    # r in (0, infinity):
    # f_c^r(x) = (1/(2*sqrt(pi*r))) * ∫_R exp(-t^2/(4r)) / (e^{-2c}cos^2 t - x e^{-c}cos t + 1) dt
    sigma = np.sqrt(2.0 * r)

    def integrand(t):
        ct = np.cos(t)
        den = np.exp(-2 * c) * ct * ct - x_val * np.exp(-c) * ct + 1.0
        if abs(den) < 1e-12:
            return 0.0
        weight = np.exp(-(t * t) / (4.0 * r)) / (2.0 * np.sqrt(np.pi * r))
        return weight / den

    L = min(4.0 * sigma, 100.0)
    try:
        val = _quad_safe(integrand, -L, L, epsabs=epsabs, epsrel=epsrel, limit=120)
        return max(val, 0.0)
    except Exception:
        t = np.linspace(-L, L, 1200)
        ct = np.cos(t)
        den = np.exp(-2 * c) * ct * ct - x_val * np.exp(-c) * ct + 1.0
        w = np.exp(-(t * t) / (4.0 * r)) / (2.0 * np.sqrt(np.pi * r))
        vals = np.where(np.abs(den) < 1e-12, 0.0, w / den)
        return max(float(np.trapezoid(vals, t)), 0.0)


def f_density(x, c, r, epsabs=1e-6, epsrel=1e-4):
    vals = np.zeros_like(x, dtype=float)
    for i, xv in enumerate(x):
        vals[i] = _f_single_x(xv, c, r, epsabs=epsabs, epsrel=epsrel)
    return vals


def eta_continuous_density(x, c, r):
    return f_density(x, c, r, epsabs=1e-5, epsrel=1e-3) * semicircle_density(x)


def sample_R_r(r, n_samples=2500):
    if r == 0.0:
        return np.ones(n_samples)
    if np.isinf(r):
        return np.cos(np.random.uniform(0.0, 2.0 * np.pi, size=n_samples))
    return np.cos(np.random.normal(0.0, np.sqrt(2.0 * r), size=n_samples))


def singular_representation(c, r, n_samples=2500):
    # This mirrors the current README formula literally.
    R = sample_R_r(r, n_samples=n_samples)
    eps = 1e-12
    masses = np.maximum(0.0, 1.0 - np.exp(2.0 * c) / (R * R + eps))
    q = np.exp(-c) * R
    atoms = q + np.exp(c) / (q + eps)

    ok = np.isfinite(atoms) & np.isfinite(masses) & (masses > 0.0)
    atoms = atoms[ok]
    masses = masses[ok]
    total_mass = float(np.mean(masses)) if len(masses) else 0.0
    return atoms, masses, total_mass


def tv_distance_to_semicircle(c, r, x_grid):
    # Implemented exactly from the x-integral formulas:
    # r in (0,∞):
    # d_TV(c,r) = (1/(2π)) ∫_{-2}^{2} |1 - f_c^r(x)| sqrt(4-x^2) dx
    # where f_c^r(x) = (1/(2√(πr))) ∫_R exp(-t^2/(4r)) / (e^{-2c}cos^2(t)-x e^{-c}cos(t)+1) dt
    #
    # r = 0:
    # d_TV(c,0) = (1/(4π)) ∫_{-2}^{2} sqrt(4-x^2) |1 - 1/(e^{-2c}-x e^{-c}+1)| dx
    #
    # r = ∞:
    # d_TV(c,∞) = (1/(4π)) ∫_{-2}^{2} sqrt(4-x^2) |1 - (1/(2π))∫_0^{2π} dt/(e^{-2c}cos^2(t)-x e^{-c}cos(t)+1)| dx
    mask = np.abs(x_grid) <= 2.0
    x_use = x_grid[mask]
    fvals = f_density(x_use, c, r, epsabs=1e-5, epsrel=1e-3)

    core = np.sqrt(np.maximum(0.0, 4.0 - x_use * x_use)) * np.abs(1.0 - fvals)

    if r == 0.0 or np.isinf(r):
        prefactor = 1.0 / (4.0 * np.pi)
    else:
        prefactor = 1.0 / (2.0 * np.pi)

    tv_val = float(prefactor * np.trapezoid(core, x_use))

    # Keep x-grid outputs for optional diagnostics/plotting
    f_full = np.zeros_like(x_grid, dtype=float)
    f_full[mask] = fvals
    nu = semicircle_density(x_grid)
    integrand = 0.5 * np.abs(1.0 - f_full) * nu
    return tv_val, f_full, integrand


x_plot = np.linspace(-2.15, 2.15, 350)
# Restrict TV curve to c >= 0 to match monotone-decay regime in practice.
c_tv_grid = np.linspace(0.0, 3.0, 121)


## Local Sliders Per Plot

Each interactive plot below has its own local sliders so you can adjust parameters without scrolling back.

Parameter ranges used in every plot:
- $c\in[-1/2,9]$
- $r\in[0,15]\cup\{\infty\}$

In [ ]:
# Plot-specific slider options/constants
PLOT1_C_MIN = -1.5
PLOT1_C_MAX = 7.0
PLOT1_C_STEP = 0.25

PLOT2_C_MIN = -7.0
PLOT2_C_MAX = 0.0
PLOT2_C_STEP = 0.25

R_OPTIONS = [(f'{v:.1f}', float(v)) for v in np.arange(0.0, 15.0 + 0.5, 0.5)] + [('∞', np.inf)]

## Plot 1: Continuous Part of $\eta_c^r$

This plot shows the Lebesgue density of the absolutely continuous part:
$$
\rho_{\mathrm{cont}}(x)=f_c^r(x)\,\mathbf{1}_{|x|\le 2}\,\frac{\sqrt{4-x^2}}{2\pi}.
$$

In [ ]:
out_cont = Output()

# Local sliders for Plot 1
c_slider_cont = FloatSlider(value=0.0, min=PLOT1_C_MIN, max=PLOT1_C_MAX, step=PLOT1_C_STEP, description='c:')
r_slider_cont = SelectionSlider(options=R_OPTIONS, value=5.0, description='r:')

def update_continuous_plot(*_):
    c_val = c_slider_cont.value
    r_val = r_slider_cont.value

    rho = eta_continuous_density(x_plot, c_val, r_val)
    dx = x_plot[1] - x_plot[0]
    cont_mass = float(np.sum(rho) * dx)

    with out_cont:
        out_cont.clear_output(wait=True)
        fig = go.Figure()
        fig.add_trace(go.Scatter(
            x=x_plot,
            y=rho,
            mode='lines',
            line=dict(color='darkred', width=3),
            fill='tozeroy',
            fillcolor='rgba(139,0,0,0.18)',
            name='continuous density',
        ))
        r_label = '∞' if np.isinf(r_val) else f'{r_val:.1f}'
        fig.update_layout(
            title=f'Continuous Part Density of η_c^r (c={c_val:.2f}, r={r_label})<br><sub>Approx. mass = {cont_mass:.6f}</sub>',
            xaxis_title='x',
            yaxis_title='density',
            template='plotly_white',
            width=980,
            height=480,
            xaxis=dict(range=[-2.2, 2.2]),
        )
        fig.show()

c_slider_cont.observe(update_continuous_plot, names='value')
r_slider_cont.observe(update_continuous_plot, names='value')

display(VBox([
    Label('Plot 1 controls: c in [-1.5, 7], r in [0,15] ∪ {∞}'),
    HBox([c_slider_cont]),
    HBox([r_slider_cont]),
    out_cont,
]))
update_continuous_plot()

## Plot 2: Singular Part (Sampling-Based Representation)

This plot keeps its own custom parameter window:
- $c\in[-15,0]$
- $r\in[0,15]\cup\{\infty\}$

It displays a sampled weighted histogram of the candidate atom locations
$$
x = e^{-c}R_r + e^cR_r^{-1},
$$
with weights
$$
\left(1-e^{2c}R_r^{-2}\right)_+.
$$
If no bar appears, that means the current numerical sampling produced no positive sampled weights for this formula.

In [ ]:
out_sing = Output()

# Local sliders for Plot 2
c_slider_sing = FloatSlider(value=-2.0, min=PLOT2_C_MIN, max=PLOT2_C_MAX, step=PLOT2_C_STEP, description='c:')
r_slider_sing = SelectionSlider(options=R_OPTIONS, value=5.0, description='r:')

def update_singular_plot(*_):
    c_val = c_slider_sing.value
    r_val = r_slider_sing.value
    atoms, masses, total_mass = singular_representation(c_val, r_val, n_samples=3000)

    with out_sing:
        out_sing.clear_output(wait=True)
        fig = go.Figure()

        if len(atoms) > 0 and total_mass > 1e-8:
            bins = np.linspace(-40, 40, 160)
            hist, edges = np.histogram(atoms, bins=bins, weights=masses)
            centers = 0.5 * (edges[:-1] + edges[1:])
            width = float(edges[1] - edges[0])
            fig.add_trace(go.Bar(
                x=centers,
                y=hist,
                width=width,
                marker=dict(color='rgba(220, 20, 60, 0.70)'),
                name='singular representation',
            ))
        else:
            fig.add_annotation(
                x=0.5, y=0.5, xref='paper', yref='paper',
                text='No sampled positive singular weight for this (c, r)',
                showarrow=False
            )

        r_label = '∞' if np.isinf(r_val) else f'{r_val:.1f}'
        fig.update_layout(
            title=f'Singular Part Representation of η_c^r (c={c_val:.2f}, r={r_label})<br><sub>Estimated sampled singular mass = {total_mass:.6f}</sub>',
            xaxis_title='atom location',
            yaxis_title='weighted histogram mass',
            template='plotly_white',
            width=980,
            height=480,
            xaxis=dict(range=[-40, 40]),
        )
        fig.show()

c_slider_sing.observe(update_singular_plot, names='value')
r_slider_sing.observe(update_singular_plot, names='value')

display(VBox([
    Label('Plot 2 controls: c in [-15, 0], r in [0,15] ∪ {∞}'),
    HBox([c_slider_sing]),
    HBox([r_slider_sing]),
    out_sing,
]))
update_singular_plot()

## Plot 3: Total Variation Distance $d_{TV}(c,r)$ (Continuous + Half Singular Weight)

This section uses the same continuous-part formula as before, and adds half of the singular weight:

$$
d_{TV}^{\mathrm{mix}}(c,r)=d_{TV}^{\mathrm{cont}}(c,r)+\frac{1}{2}\,m_{\mathrm{sing}}(c,r),
$$
where
$$
m_{\mathrm{sing}}(c,r)=\mathbb E\!\left[\left(1-e^{2c}R_r^{-2}\right)_+\right].
$$

For the continuous part,

For $r\in(0,\infty)$,
$$
d_{TV}^{\mathrm{cont}}(c,r)=\frac{1}{4\pi}\int_{-2}^{2}\left|1-\frac{1}{2\sqrt{\pi r}}\int_{\mathbb{R}}\frac{e^{-t^{2}/(4r)}}{e^{-2c}\cos^{2}(t)-x e^{-c}\cos(t)+1}\,dt\right|\sqrt{4-x^{2}}\,dx.
$$

For $r=0$,
$$
d_{TV}^{\mathrm{cont}}(c,0)=\frac{1}{4\pi}\int_{-2}^{2}\sqrt{4-x^{2}}\left|1-\frac{1}{e^{-2c}-x e^{-c}+1}\right|\,dx.
$$

For $r=\infty$,
$$
d_{TV}^{\mathrm{cont}}(c,\infty)=\frac{1}{4\pi}\int_{-2}^{2}\sqrt{4-x^{2}}\left|1-\frac{1}{2\pi}\int_{0}^{2\pi}\frac{dt}{e^{-2c}\cos^{2}(t)-x e^{-c}\cos(t)+1}\right|\,dx.
$$

We plot $c\mapsto d_{TV}^{\mathrm{mix}}(c,r)$ on $c\in[-3,3]$ with y-axis fixed to $[0,1]$.

In [ ]:
out_tv = Output()

# Slider for r (plot is c -> d_TV(c,r))
r_slider_tv = SelectionSlider(options=R_OPTIONS, value=5.0, description='r:')

# c-grid requested by user
c_tv_grid = np.linspace(-3.0, 3.0, 121)

# Cache curves for previously selected r values (slider values are discrete)
_tv_curve_cache = {}


def _safe_denominator(den, eps=1e-12):
    """Avoid division blow-ups in near-singular denominator evaluations."""
    sign = np.where(den >= 0.0, 1.0, -1.0)
    return np.where(np.abs(den) < eps, sign * eps, den)


def _f_vals_for_x_grid(x_vals, c, r):
    """Compute f_c^r(x) on an x-grid using the three requested regimes."""
    exp_m2c = np.exp(-2.0 * c)
    exp_mc = np.exp(-c)

    # r = 0 exact formula
    if r == 0.0:
        den = exp_m2c - x_vals * exp_mc + 1.0
        den = _safe_denominator(den)
        return 1.0 / den

    # r = infinity: average over t in [0, 2pi]
    if np.isinf(r):
        t = np.linspace(0.0, 2.0 * np.pi, 1601)
        cos_t = np.cos(t)
        base = exp_m2c * (cos_t ** 2) + 1.0
        den = base[:, None] - exp_mc * cos_t[:, None] * x_vals[None, :]
        den = _safe_denominator(den)
        integrand = 1.0 / den
        return (1.0 / (2.0 * np.pi)) * np.trapezoid(integrand, t, axis=0)

    # r in (0, infinity): Gaussian weighted integral on R
    sigma = np.sqrt(2.0 * r)
    L = min(6.0 * sigma, 150.0)
    t = np.linspace(-L, L, 2001)
    cos_t = np.cos(t)
    base = exp_m2c * (cos_t ** 2) + 1.0
    den = base[:, None] - exp_mc * cos_t[:, None] * x_vals[None, :]
    den = _safe_denominator(den)

    weight = np.exp(-(t ** 2) / (4.0 * r)) / (2.0 * np.sqrt(np.pi * r))
    integrand = weight[:, None] / den
    return np.trapezoid(integrand, t, axis=0)


def _singular_mass(c, r):
    """
    m_sing(c,r) = E[(1 - e^(2c) / R_r^2)_+], with R_r = cos(T_r).
    This contributes as +0.5 * m_sing(c,r) in the mixed TV curve.
    """
    ec2 = np.exp(2.0 * c)

    if r == 0.0:
        return max(0.0, 1.0 - ec2)

    if np.isinf(r):
        t = np.linspace(0.0, 2.0 * np.pi, 4001)
        cos2 = np.cos(t) ** 2
        vals = np.where(cos2 > 1e-12, np.maximum(0.0, 1.0 - ec2 / cos2), 0.0)
        return float(np.trapezoid(vals, t) / (2.0 * np.pi))

    sigma = np.sqrt(2.0 * r)
    L = min(8.0 * sigma, 180.0)
    t = np.linspace(-L, L, 4001)
    cos2 = np.cos(t) ** 2
    vals = np.where(cos2 > 1e-12, np.maximum(0.0, 1.0 - ec2 / cos2), 0.0)
    weight = np.exp(-(t ** 2) / (4.0 * r)) / (2.0 * np.sqrt(np.pi * r))
    return float(np.trapezoid(weight * vals, t))


def tv_curve_from_formula(r):
    """
    Compute c -> d_TV^mix(c,r) where
    d_TV^mix = d_TV^cont + 0.5 * m_sing.

    Numerically stable equivalent for continuous term uses x = 2 cos(theta):
    (1/(4pi)) * int_{-2}^2 sqrt(4-x^2) |1-f(x)| dx
    = (1/pi) * int_0^pi sin(theta)^2 |1-f(2cos(theta))| dtheta.
    """
    cache_key = 'inf' if np.isinf(r) else float(r)
    if cache_key in _tv_curve_cache:
        return _tv_curve_cache[cache_key]

    theta = np.linspace(1e-6, np.pi - 1e-6, 501)
    x_vals = 2.0 * np.cos(theta)
    sin2 = np.sin(theta) ** 2

    curve = []
    for c in c_tv_grid:
        f_vals = _f_vals_for_x_grid(x_vals, c, r)
        tv_integrand_theta = (1.0 / np.pi) * np.abs(1.0 - f_vals) * sin2
        tv_cont = float(np.trapezoid(tv_integrand_theta, theta))
        m_sing = _singular_mass(c, r)
        tv_mix = tv_cont + 0.5 * m_sing
        curve.append(tv_mix)

    curve = np.array(curve)
    _tv_curve_cache[cache_key] = curve
    return curve


def update_tv_plot(*_):
    r_val = r_slider_tv.value
    tv_values = tv_curve_from_formula(r_val)

    with out_tv:
        out_tv.clear_output(wait=True)

        fig = go.Figure()
        fig.add_trace(go.Scatter(
            x=c_tv_grid,
            y=tv_values,
            mode='lines',
            line=dict(color='darkblue', width=3),
            fill='tozeroy',
            fillcolor='rgba(0, 0, 139, 0.15)',
            name='d_TV^mix(c, r)'
        ))

        r_label = '∞' if np.isinf(r_val) else f'{r_val:.1f}'
        fig.update_layout(
            title=f'd_TV^mix(c,r) = continuous + 0.5 * singular weight (r={r_label})',
            xaxis_title='c',
            yaxis_title='d_TV^mix(c,r)',
            template='plotly_white',
            width=980,
            height=480,
            xaxis=dict(range=[-3.0, 3.0]),
            yaxis=dict(range=[0.0, 1.0]),
        )
        fig.show()


r_slider_tv.observe(update_tv_plot, names='value')

display(VBox([
    Label('Plot 3 control: choose r; graph shows mixed TV = continuous + 0.5 * singular weight on c in [-3,3].'),
    HBox([r_slider_tv]),
    out_tv,
]))

update_tv_plot()

In [9]:
# Diagnostic: locate where r=0 curve has positive slope

def tv_r0_theta_stable(c):
    theta = np.linspace(1e-6, np.pi - 1e-6, 4000)
    den = np.exp(-2*c) - 2.0 * np.exp(-c) * np.cos(theta) + 1.0
    f0 = np.where(np.abs(den) < 1e-14, 0.0, 1.0 / den)
    integrand = (1.0 / np.pi) * np.abs(1.0 - f0) * (np.sin(theta) ** 2)
    return float(np.trapezoid(integrand, theta))

curve_r0_theta = np.array([tv_r0_theta_stable(c) for c in c_tv_grid])
deltas = np.diff(curve_r0_theta)
idx = int(np.argmax(deltas))

print(f"max positive delta = {deltas[idx]:.6e}")
print(f"between c={c_tv_grid[idx]:.3f} and c={c_tv_grid[idx+1]:.3f}")
print(f"values: {curve_r0_theta[idx]:.6f} -> {curve_r0_theta[idx+1]:.6f}")

print("first 8 deltas near the max region:")
start = max(0, idx-3)
end = min(len(deltas), idx+5)
for j in range(start, end):
    print(f"  [{c_tv_grid[j]:+.2f},{c_tv_grid[j+1]:+.2f}] : {deltas[j]:+.6e}")

max positive delta = 2.762826e-02
between c=-0.050 and c=0.000
values: 0.385868 -> 0.413496
first 8 deltas near the max region:
  [-0.20,-0.15] : +1.286042e-02
  [-0.15,-0.10] : +1.737860e-02
  [-0.10,-0.05] : +2.229000e-02
  [-0.05,+0.00] : +2.762826e-02
  [+0.00,+0.05] : -1.915102e-02
  [+0.05,+0.10] : -1.836376e-02
  [+0.10,+0.15] : -1.759287e-02
  [+0.15,+0.20] : -1.684135e-02
